# AI Try-On Backend - Google Colab Setup

This notebook sets up the try-on backend project in Google Colab for ML model development and testing.

## Features
- Install all dependencies
- Clone/download project from GitHub
- Setup ML models with GPU support
- Test ML inference
- Optional: Run API server with ngrok


## Step 1: Install Dependencies

Install all required packages for ML models and backend.


In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0

# Install PyTorch with CUDA (for GPU support)
# Using 2.3.0+cu118 as 2.1.0 is no longer available
%pip install -q torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu118

# Install ML and image processing libraries
%pip install -q numpy==1.24.3 opencv-python==4.8.1.78 Pillow==10.1.0 scipy==1.11.4
%pip install -q imageio==2.31.5 scikit-image==0.22.0

# Install FastAPI and web framework
%pip install -q fastapi==0.104.1 uvicorn[standard]==0.24.0 python-multipart==0.0.6

# Install Pydantic and validation
%pip install -q pydantic==2.5.0 pydantic-settings==2.1.0

# Install database (SQLite for Colab, no PostgreSQL needed)
%pip install -q sqlalchemy==2.0.23 alembic==1.12.1

# Install logging and utilities
%pip install -q structlog==23.2.0 python-dotenv==1.0.0
%pip install -q requests==2.31.0 tqdm==4.66.1 httpx==0.25.2

# Install monitoring (optional)
%pip install -q prometheus-client==0.19.0

# Install ngrok for exposing API
%pip install -q pyngrok

# Note: We skip Redis, Celery, PostgreSQL, MinIO for Colab
# Use SQLite for database and direct function calls instead of Celery

print("✅ All dependencies installed!")


## Step 1b: Verify Dependencies

Check that all required packages are installed.


In [ ]:
# Verify critical dependencies
import importlib

required_packages = [
    "torch",
    "numpy",
    "PIL",
    "cv2",
    "fastapi",
    "uvicorn",
    "pydantic",
    "sqlalchemy",
    "structlog",
    "requests",
]

missing = []
for package in required_packages:
    try:
        importlib.import_module(package)
        print(f"✅ {package}")
    except ImportError:
        missing.append(package)
        print(f"❌ {package} - MISSING")

if missing:
    print(f"\n⚠️  Missing packages: {missing}")
    print("Please re-run the installation cell above.")
else:
    print("\n✅ All required packages are installed!")


## Step 2: Clone Project

Clone the repository from GitHub.


In [ ]:
import os
import sys
from pathlib import Path

# Clone repository
repo_url = "https://github.com/StefanusSimandjuntak111/try-on-backend.git"
project_dir = "/content/try-on-backend"

if os.path.exists(project_dir):
    print(f"📁 Project already exists at {project_dir}")
    print("🔄 Updating repository...")
    !cd {project_dir} && git pull
else:
    print(f"📥 Cloning repository to {project_dir}...")
    !git clone {repo_url} {project_dir}

# Add to Python path
sys.path.insert(0, project_dir)
os.chdir(project_dir)

print(f"✅ Project cloned to {project_dir}")
print(f"📂 Current directory: {os.getcwd()}")


## Step 3: Setup Environment

Configure environment variables for Colab.


In [ ]:
import torch
import os

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set environment variables for Colab
os.environ["ML_DEVICE"] = device
os.environ["DATABASE_URL"] = "sqlite:///./test.db"
os.environ["USE_LOCAL_STORAGE"] = "true"  # Use local file storage instead of S3/MinIO
os.environ["REDIS_URL"] = "redis://localhost:6379/0"  # Not used but needed for config
os.environ["S3_ENDPOINT"] = "http://localhost:8500"  # Not used but needed for config

# Create directories
weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)
print(f"📁 Weights directory: {weights_dir.absolute()}")

uploads_dir = Path("uploads")
uploads_dir.mkdir(exist_ok=True)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

storage_dir = Path("storage")
storage_dir.mkdir(exist_ok=True)

print("✅ Environment setup complete!")
print("ℹ️  Using local file storage (no S3/MinIO needed)")


## Step 4: Test ML Models

Test loading and using ML models.


In [ ]:
from app.config import settings

# Update settings for Colab
settings.ML_DEVICE = device

print("🧪 Testing ML models...")

# Test U2-Net (Background Removal)
try:
    from app.ml.background.model import get_u2net_model
    u2net = get_u2net_model()
    print(f"✅ U2-Net loaded (device: {u2net.device})")
except Exception as e:
    print(f"⚠️  U2-Net: {str(e)}")

# Test SCHP (Human Parsing)
try:
    from app.ml.parsing.model import get_schp_model
    schp = get_schp_model()
    print(f"✅ SCHP loaded (device: {schp.device})")
except Exception as e:
    print(f"⚠️  SCHP: {str(e)}")

# Test OpenPose
try:
    from app.ml.pose.model import get_openpose_model
    openpose = get_openpose_model()
    print(f"✅ OpenPose loaded (device: {openpose.device})")
except Exception as e:
    print(f"⚠️  OpenPose: {str(e)}")

# Test HR-VITON
try:
    from app.ml.hrviton.model import get_hrviton_model
    hrviton = get_hrviton_model()
    print(f"✅ HR-VITON loaded (device: {hrviton.device})")
except Exception as e:
    print(f"⚠️  HR-VITON: {str(e)}")

print("\n✅ ML model testing complete!")


## Step 5: Run API Server (Optional)

**IMPORTANT:** Start the server FIRST, then connect ngrok. The server must be running before ngrok can connect to it.


In [ ]:
# Start FastAPI server FIRST
import subprocess
import threading
import time
import requests
from multiprocessing import Process

def run_server():
    """Run FastAPI server in a separate process."""
    import os
    os.environ["ML_DEVICE"] = device
    os.environ["DATABASE_URL"] = "sqlite:///./test.db"
    
    # Run uvicorn server
    import uvicorn
    uvicorn.run(
        "app.main:app",
        host="0.0.0.0",
        port=8500,
        log_level="info"
    )

# Start server in background process
print("🚀 Starting FastAPI server...")
server_process = Process(target=run_server, daemon=True)
server_process.start()

# Wait for server to start (check if it's responding)
print("⏳ Waiting for server to start...")
max_attempts = 30
for i in range(max_attempts):
    try:
        response = requests.get("http://localhost:8500/", timeout=2)
        if response.status_code == 200:
            print("✅ Server is running!")
            break
    except:
        if i < max_attempts - 1:
            time.sleep(1)
        else:
            print("❌ Server failed to start. Check for errors above.")
            raise Exception("Server did not start within 30 seconds")

print(f"📡 Server running at http://localhost:8500")
print(f"📚 API Docs: http://localhost:8500/docs")


## Troubleshooting

If you get connection errors:

1. **Check if server is running:**
```python
import requests
response = requests.get("http://localhost:8500/")
print(f"Status: {response.status_code}")
```

2. **Check if port 8500 is in use:**
```python
!lsof -ti:8500 || echo "Port 8500 is free"
```

3. **Kill existing processes:**
```python
!pkill -f uvicorn || echo "No uvicorn processes found"
```

4. **Restart server:**
   - Stop the server cell above
   - Restart and run it again
   - Wait for "✅ Server is running!" message
   - Then run ngrok cell


In [ ]:
# NOW connect ngrok (AFTER server is running)
from pyngrok import ngrok

# Setup ngrok (get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "352jfEHhZz5x7mEkDTvoJdoRsxS_4iZBggqou4VdUPEx2N1Ge"  # Set your token here if you have one
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    print("✅ ngrok authenticated")
else:
    print("ℹ️  Using ngrok free tier (no auth token)")

# Verify server is still running before connecting
try:
    response = requests.get("http://localhost:8500/", timeout=2)
    if response.status_code == 200:
        # Start ngrok tunnel
        print("🔗 Connecting ngrok...")
        public_url = ngrok.connect(8500, bind_tls=True)
        print(f"🌐 Public URL: {public_url}")
        print(f"📚 API Docs: {public_url}/docs")
        print(f"🔒 HTTPS: {public_url.replace('http://', 'https://')}")
    else:
        print("❌ Server is not responding. Please restart the server cell above.")
except Exception as e:
    print(f"❌ Cannot connect to server: {e}")
    print("⚠️  Please make sure the server is running in the cell above.")
